In [5]:
import pandas as pd
df = pd.read_csv("imdb_anime.csv")
df.head()

,Title,Genre,User Rating,Number of Votes,Runtime,Year,Summary,Stars,Certificate,Metascore,Gross,Episode,Episode Title
0,One Piece,"Animation, Action, Adventure",8.9,"187,689",24 min,(1999– ),Follows the adventures of Monkey D. Luffy and ...,"Mayumi Tanaka,Laurent Vernin,Akemi Okamura,Ton...",TV-14,NaN,187689,0,NaN
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,"Animation, Action, Adventure",7.4,"28,895",99 min,(2023),The film follows the Turtle brothers as they w...,NaN,PG,74,28895,0,NaN
2,The Super Mario Bros. Movie,"Animation, Adventure, Comedy",7.1,"189,108",92 min,(2023),A plumber named Mario travels through an under...,NaN,PG,46,189108,0,NaN
3,Attack on Titan,"Animation, Action, Adventure",9.1,"434,457",24 min,(2013–2023),After his hometown is destroyed and his mother...,"Josh Grelle,Bryce Papenbrook,Yûki Kaji,Yui Ish...",TV-MA,NaN,434457,0,NaN
4,Jujutsu Kaisen,"Animation, Action, Adventure",8.5,"82,909",24 min,(2020– ),A boy swallows a cursed talisman - the finger ...,"Junya Enoki,Yûichi Nakamura,Adam McArthur,Yuma...",TV-MA,NaN,82909,0,NaN


In [6]:
df.shape #45,717 anime titles and 13 columns

(45717, 13)

In [7]:
df.columns.tolist()

['Title',
 'Genre',
 'User Rating',
 'Number of Votes',
 'Runtime',
 'Year',
 'Summary',
 'Stars',
 'Certificate',
 'Metascore',
 'Gross',
 'Episode',
 'Episode Title']

In [8]:
df.isnull().sum() #This counts how many missing values exist in each column.

Title                  0
Genre                  0
User Rating        20708
Number of Votes    20708
Runtime            13168
Year                 126
Summary            22170
Stars              32041
Certificate        17023
Metascore          45376
Gross              20708
Episode                0
Episode Title      10807
dtype: int64

In [9]:
df.iloc[0] # sample row

Title                                                      One Piece
Genre                                   Animation, Action, Adventure
User Rating                                                      8.9
Number of Votes                                              187,689
Runtime                                                       24 min
Year                                                        (1999– )
Summary            Follows the adventures of Monkey D. Luffy and ...
Stars              Mayumi Tanaka,Laurent Vernin,Akemi Okamura,Ton...
Certificate                                                    TV-14
Metascore                                                        NaN
Gross                                                         187689
Episode                                                            0
Episode Title                                                    NaN
Name: 0, dtype: str

In [13]:
df['Episode'].value_counts().head(10) #For our recommender we only want series-level rows (Episode = 0), not individual episodes. 
# Otherwise One Piece would appear 1000+ times.
# Run this to see how many series vs episode rows we have:

Episode
1          34909
0          10807
Episode        1
Name: count, dtype: int64

In [11]:
series_df=df[df['Episode']==0]
print(series_df.shape)
series_df.head(3)

(0, 13)


,Title,Genre,User Rating,Number of Votes,Runtime,Year,Summary,Stars,Certificate,Metascore,Gross,Episode,Episode Title


In [12]:
series_df=df[df['Episode']=='0'] # How many unique anime titles do we have at series level? — creates a True/False filter for rows where Episode is 0
print(series_df.shape) # keeps only the rows where that filter is True. We save it as series_df — our cleaner starting point
series_df.head(3)

(10807, 13)


,Title,Genre,User Rating,Number of Votes,Runtime,Year,Summary,Stars,Certificate,Metascore,Gross,Episode,Episode Title
0,One Piece,"Animation, Action, Adventure",8.9,"187,689",24 min,(1999– ),Follows the adventures of Monkey D. Luffy and ...,"Mayumi Tanaka,Laurent Vernin,Akemi Okamura,Ton...",TV-14,NaN,187689,0,NaN
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,"Animation, Action, Adventure",7.4,"28,895",99 min,(2023),The film follows the Turtle brothers as they w...,NaN,PG,74,28895,0,NaN
2,The Super Mario Bros. Movie,"Animation, Adventure, Comedy",7.1,"189,108",92 min,(2023),A plumber named Mario travels through an under...,NaN,PG,46,189108,0,NaN


``` Start cleaning ```

In [14]:
import re
# 1. Keep only series-level rows
df_clean = df[df['Episode'] == '0'].copy()

# 2. Drop columns that are too empty to be useful
df_clean = df_clean.drop(columns=['Metascore', 'Gross', 'Stars', 'Certificate', 'Summary', 'Episode', 'Episode Title'])

# 3. Clean 'Number of Votes' - remove commas, convert to number
df_clean['Number of Votes'] = df_clean['Number of Votes'].str.replace(',', '').astype(float)

# 4. Clean 'Runtime' - extract just the number
df_clean['Runtime'] = df_clean['Runtime'].str.extract(r'(\d+)').astype(float)

# 5. Clean 'Year' - extract the first 4-digit year
df_clean['Year'] = df_clean['Year'].str.extract(r'(\d{4})').astype(float)

# 6. Clean 'User Rating' - convert to number
df_clean['User Rating'] = pd.to_numeric(df_clean['User Rating'], errors='coerce')

# Check result

print(df_clean.shape)
df_clean.isnull().sum() 
    

(10807, 6)


Title                 0
Genre                 0
User Rating        3054
Number of Votes    3054
Runtime            2701
Year                145
dtype: int64

In [15]:
# Drop rows with no User Rating (we need this for the recommender)
df_clean = df_clean.dropna(subset=['User Rating'])

# Fill missing Runtime and Year with the median (middle value)
df_clean['Runtime']= df_clean['Runtime'].fillna(df_clean['Runtime'].median())
df_clean['Year'] = df_clean['Year'].fillna(df_clean['Year'].median())

# Check result
print(df_clean.shape)
df_clean.isnull().sum()

(7753, 6)


Title              0
Genre              0
User Rating        0
Number of Votes    0
Runtime            0
Year               0
dtype: int64

In [16]:
df_clean.describe()

,User Rating,Number of Votes,Runtime,Year
count,7753.000000,7.753000e+03,7753.000000,7753.000000
mean,6.805314,8.686098e+03,43.453631,2004.627112
std,0.963581,5.638453e+04,45.500280,16.217827
min,1.000000,5.000000e+00,1.000000,1907.000000
25%,6.200000,4.000000e+01,24.000000,1997.000000
50%,6.900000,2.070000e+02,25.000000,2009.000000
75%,7.500000,1.328000e+03,54.000000,2016.000000
max,9.800000,1.162284e+06,780.000000,2023.000000


Transformation

In [17]:
import numpy as np
# 1. Log-scale NUmber of Votes (reduces the huge gap between popular and obscure anime)
df_clean['Votes Log'] = np.log1p(df_clean['Number of Votes'])

# 2. Normalize User Rating and Votes Log to 0-1 scale
df_clean['Rating Norm'] = (df_clean['User Rating'] - df_clean['User Rating'].min()) / (df_clean['User Rating'].max() - df_clean['User Rating'].min())
df_clean['Votes Norm'] = (df_clean['Votes Log'] - df_clean['Votes Log'].min()) / (df_clean['Votes Log'].max() - df_clean['Votes Log'].min())

df_clean[['Title', 'User Rating', 'Rating Norm', 'Number of Votes', 'Votes Log', 'Votes Norm']].head()

                                                                                  


,Title,User Rating,Rating Norm,Number of Votes,Votes Log,Votes Norm
0,One Piece,8.9,0.897727,187689.0,12.142547,0.850227
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,7.4,0.727273,28895.0,10.271458,0.696534
2,The Super Mario Bros. Movie,7.1,0.693182,189108.0,12.150079,0.850846
3,Attack on Titan,9.1,0.920455,434457.0,12.981855,0.919169
4,Jujutsu Kaisen,8.5,0.852273,82909.0,11.325511,0.783115


Feature Engineering 

In [18]:
# 1. Popularity score - combines rating and vote count
df_clean['Popularity Score'] = 0.6 * df_clean['Rating Norm'] + 0.4 * df_clean['Votes Norm']

# 2. Split genres into a list then one-hot encode them
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

print("Genre columns created:", genre_dummies.columns.tolist())
print("Shape:", genre_dummies.shape)
genre_dummies.head(3)

Genre columns created: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War']
Shape: (7753, 22)


,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Short,Sport,Thriller,War
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Combine into final feature matrix

In [19]:
# Combine genre columns with our numeric features
feature_df = pd.concat([df_clean[['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm']].reset_index(drop=True)], axis=1)

print(feature_df.shape)
feature_df.head(3)


(7753, 4)


,Title,Popularity Score,Rating Norm,Votes Norm
0,One Piece,0.878727,0.897727,0.850227
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,0.714977,0.727273,0.696534
2,The Super Mario Bros. Movie,0.756248,0.693182,0.850846


In [20]:
# Reset index on df_clean first, then rebuild
df_clean = df_clean.reset_index(drop=True)
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

feature_df = pd.concat([
    df_clean[['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm']],
    genre_dummies
], axis=1)

print(feature_df.shape)
print(feature_df.columns.tolist())

(7753, 26)
['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm', 'Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War']


Recommendation Engine

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

# Build the numeric matrix (everything except Title)
feature_matrix = feature_df.drop(columns=['Title']).values

# Compute similarity between every pair of anime
similarity_matrix = cosine_similarity(feature_matrix)

print(similarity_matrix.shape)

(7753, 7753)


In [23]:
def recommend(title, n=10):
    # Find the row number for this anime
    matches = feature_df[feature_df['Title'].str.lower() == title.lower()]

    if matches.empty:
        print(f"'{title}' not found. Check the spelling.")
        return

    idx = matches.index[0]

    # Get similarity scores for this anime against all others
    scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity, highest first, skip the first one (itself)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]

    # Build result table
    results = []
    for i, score in scores:
        results.append({
            'Title': feature_df.iloc[i]['Title'],
            'Similarity': round(score, 3),
            'Rating': round(df_clean.iloc[i]['User Rating'], 1),
            'Genres': df_clean.iloc[i]['Genre']
        })

    return pd.DataFrame(results)

# Test it
recommend("Attack on Titan")

,Title,Similarity,Rating,Genres
0,Attack on Titan,1.000,9.1,"Animation, Action, Adventure"
1,Fullmetal Alchemist: Brotherhood,1.000,9.1,"Animation, Action, Adventure"
2,Fullmetal Alchemist: Brotherhood,1.000,9.1,"Animation, Action, Adventure"
3,One Piece,1.000,8.9,"Animation, Action, Adventure"
4,One Piece,1.000,8.9,"Animation, Action, Adventure"
5,Princess Mononoke,0.999,8.3,"Animation, Action, Adventure"
6,Princess Mononoke,0.999,8.3,"Animation, Action, Adventure"
7,Dragon Ball Z,0.999,8.8,"Animation, Action, Adventure"
8,Dragon Ball Z,0.999,8.8,"Animation, Action, Adventure"
9,Naruto: Shippuden,0.999,8.7,"Animation, Action, Adventure"


In [24]:
# Fix 1: Remove duplicate titles, keep the one with most votes
df_clean = df_clean.sort_values('Number of Votes', ascending=False)
df_clean = df_clean.drop_duplicates(subset='Title', keep='first')
df_clean = df_clean.reset_index(drop=True)

# Rebuild feature matrix with deduplicated data
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

feature_df = pd.concat([
    df_clean[['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm']],
    genre_dummies
], axis=1).reset_index(drop=True)

# Fix 2: Give more weight to rating and votes so similar-genre anime are ranked by quality
feature_df['Rating Norm'] = feature_df['Rating Norm'] * 3
feature_df['Votes Norm'] = feature_df['Votes Norm'] * 2

# Rebuild similarity matrix
feature_matrix = feature_df.drop(columns=['Title']).values
similarity_matrix = cosine_similarity(feature_matrix)

print(f"Unique anime: {len(df_clean)}")
recommend("Attack on Titan")

Unique anime: 6808


,Title,Similarity,Rating,Genres
0,One Piece,1.000,8.9,"Animation, Action, Adventure"
1,Fullmetal Alchemist: Brotherhood,0.999,9.1,"Animation, Action, Adventure"
2,Naruto: Shippuden,0.999,8.7,"Animation, Action, Adventure"
3,Dragon Ball Z,0.999,8.8,"Animation, Action, Adventure"
4,Demon Slayer: Kimetsu no Yaiba,0.999,8.6,"Animation, Action, Adventure"
5,Cowboy Bebop,0.999,8.9,"Animation, Action, Adventure"
6,Naruto,0.999,8.4,"Animation, Action, Adventure"
7,Hunter x Hunter,0.999,9.0,"Animation, Action, Adventure"
8,Princess Mononoke,0.999,8.3,"Animation, Action, Adventure"
9,Jujutsu Kaisen,0.999,8.5,"Animation, Action, Adventure"


In [25]:
recommend("Spirited Away")

,Title,Similarity,Rating,Genres
0,Howl's Moving Castle,1.000,8.2,"Animation, Adventure, Family"
1,WALL·E,1.000,8.4,"Animation, Adventure, Family"
2,Castle in the Sky,0.999,8.0,"Animation, Adventure, Family"
3,The Little Mermaid,0.998,7.6,"Animation, Adventure, Family"
4,Treasure Planet,0.997,7.2,"Animation, Adventure, Family"
5,Mary and the Witch's Flower,0.989,6.8,"Animation, Adventure, Family"
6,The Hobbit,0.988,6.7,"Animation, Adventure, Family"
7,Adventures of the Gummi Bears,0.988,7.5,"Animation, Adventure, Family"
8,Voltron: Defender of the Universe,0.981,7.9,"Animation, Adventure, Family"
9,Penguin Highway,0.979,7.1,"Animation, Adventure, Family"


In [26]:
recommend("Sailor Moon")


,Title,Similarity,Rating,Genres
0,Overlord,1.0,7.7,"Animation, Action, Adventure"
1,Berserk: The Golden Age Arc I - The Egg of the...,1.0,7.5,"Animation, Action, Adventure"
2,Tekkonkinkreet,1.0,7.5,"Animation, Action, Adventure"
3,Miraculous: Tales of Ladybug & Cat Noir,1.0,7.5,"Animation, Action, Adventure"
4,Berserk: The Golden Age Arc II - The Battle fo...,1.0,7.7,"Animation, Action, Adventure"
5,The Last: Naruto the Movie,1.0,7.6,"Animation, Action, Adventure"
6,Soul Eater,1.0,7.8,"Animation, Action, Adventure"
7,TaleSpin,1.0,7.5,"Animation, Action, Adventure"
8,Afro Samurai,1.0,7.6,"Animation, Action, Adventure"
9,Goblin Slayer,1.0,7.5,"Animation, Action, Adventure"
